In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, concat_ws, split,to_date,try_to_date
from pyspark.sql import functions as F


In [0]:
df= spark.table("novacart_catalog.001_bronze.customers")
display(df.limit(200))

In [0]:
df = df.withColumn("customer_name", F.trim(F.col("customer_name"))) \
       .withColumn("country_code", F.trim(F.col("country_code"))) \
       .withColumn("channel", F.trim(F.col("channel")))

display(df.limit(200))

In [0]:
null_values = ["", "null", "NULL", "\\N", "-", "?", ","]

df = df.withColumn(
    "customer_name",
    F.when(
        (F.trim(F.col("customer_name")).isin(null_values)) | F.col("customer_name").isNull(),
        F.lit("unknown")
    ).otherwise(F.col("customer_name"))
)
df.display()

In [0]:
null_values = ["", "null", "NULL", "\\N", "-", "?", ","]

df = df.withColumn(
    "email",
    F.when(
        (F.trim(F.col("email")).isin(null_values)) | F.col("email").isNull(),
        F.lit("unknown")
    ).otherwise(F.col("email"))
)
df.display()

In [0]:
df = df.withColumn(
    "registration_date",
    F.coalesce(
        F.expr("try_to_date(registration_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(registration_date, 'd/M/yyyy')"),
        F.expr("try_to_date(registration_date, 'M/d/yyyy')"),
        F.expr("try_to_date(registration_date, 'MM-dd-yyyy')"),
       F.col("registration_date")
    )
)
display(df.limit(200))

In [0]:

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("novacart_catalog.002_silver.customers")
